# 05 · Random forest — Toronto

City RF on the **shared candidate list** from `05_feature_selection.ipynb` (same inputs as
the forest there / the reference RF in `04_baseline.ipynb`). Template for the other
`05_rf_*.ipynb` city notebooks.

Setup matches 04/05:
- Sept 2026 snapshot via `strfair.prepare()` (includes Kandice's review sentiment columns)
- drop `sentiment_review_count`; add `no_sentiment` for listings with fewer than 3 usable reviews
- GroupKFold(5) by host; MAE/RMSE on log price plus MAE in dollars
- untuned RF: 300 trees, `min_samples_leaf=5` (same as 04_baseline)

Before the first run (from the repo root):

```bash
python scripts/download_data.py toronto@2026-09-16
```

First `prepare()` per city is slow (sentiment scoring). Later runs read the cache.


In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="Pandas requires version")
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "strfair").is_dir())))
import strfair

CITY = "toronto"
SNAPSHOT = "2026-09-16"  # same Sept snapshot as 04_baseline / 05_feature_selection


In [ ]:
data = strfair.prepare(CITY, SNAPSHOT)   # compare the `data` hash with a teammate

## Feature matrix

Use the RF shared-candidate list from `05_feature_selection` (80 columns; near-constant
amenities / flags already dropped there). Keep only columns present for this city.
`checkin_dow` / `checkin_month` are treated as categories. Trees: median-impute numerics,
ordinal-encode categoricals (same as 04_baseline RF).

In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OrdinalEncoder

# Shared RF candidates from 05_feature_selection (includes sentiment + no_sentiment;
# sentiment_review_count already excluded). neighbourhood_group may be absent in some cities.
RF_FEATURES = [
    "lead_days", "quote_nights", "weekend_checkin", "checkin_dow", "checkin_month",
    "is_monthly_quote", "lead_band", "has_discount", "discount_share", "minimum_nights",
    "min_nights_band", "room_type", "room", "property_group", "accommodates", "bedrooms",
    "beds", "shared_bath", "baths", "guests_per_bedroom", "guests_per_bed", "latitude",
    "longitude", "km_to_centre", "log_km_to_centre", "neighbourhood", "neighbourhood_group",
    "superhost", "host_listings_count", "calculated_host_listings_count",
    "calculated_host_listings_count_entire_homes", "calculated_host_listings_count_private_rooms",
    "log_host_listings", "multi_host", "single_listing_host", "host_entire_home_share",
    "host_years", "number_of_reviews", "log_reviews", "no_reviews", "tenure_months",
    "new_listing", "months_since_last_review", "review_scores_rating", "review_scores_accuracy",
    "review_scores_cleanliness", "review_scores_checkin", "review_scores_communication",
    "review_scores_location", "no_rating", "rating_filled", "name_len", "desc_len",
    "kw_luxury", "kw_view", "kw_renovated", "kw_cozy", "kw_walkable", "has_license",
    "license_exempt", "host_about_len", "maximum_nights", "n_amenities", "amg_tv_streaming",
    "amg_entertainment", "amg_outdoor_views", "amg_grill", "amg_luxury_recreation",
    "amg_cooking_appliances", "amg_climate_control", "amg_exercise_facilities", "amg_parking",
    "amg_laundry", "amg_fridge_freezer", "am_free_parking", "avg_sentiment", "sentiment_std",
    "negative_review_pct", "weighted_sentiment", "no_sentiment",
]

F = data.features.copy()
F["no_sentiment"] = F["avg_sentiment"].isna().astype(int)  # 1 = fewer than 3 usable reviews
for col in ("checkin_dow", "checkin_month"):
    if col in F.columns:
        F[col] = F[col].astype("object")

X_cols = [c for c in RF_FEATURES if c in F.columns]
missing = [c for c in RF_FEATURES if c not in F.columns]
cat = list(F[X_cols].select_dtypes(exclude="number").columns)
num = [c for c in X_cols if c not in cat]

y = F["log_price"]
X = F[X_cols]
groups = data.sample["host_id"].values  # GroupKFold: one host stays in one fold

print(f"{len(X_cols)} inputs ({len(num)} numeric, {len(cat)} categorical) | n={len(F):,}")
if missing:
    print("RF_FEATURES missing in this city:", missing)
print("categoricals:", cat)
print("sentiment cols present:", [c for c in X_cols if "sentiment" in c])

## Model + 5-fold out-of-fold predictions

Untuned RF matching `04_baseline`: 300 trees, `min_samples_leaf=5`. `GroupKFold(5)` by
host so one host's listings never split across train and test (same protocol as 04/05).

In [ ]:
pre = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num),
    ("cat", make_pipeline(
        SimpleImputer(strategy="most_frequent"),
        OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
    ), cat),
])

rf = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42,
)

model = make_pipeline(pre, rf)
gkf = GroupKFold(n_splits=5)

oof = cross_val_predict(model, X, y, cv=gkf, groups=groups, n_jobs=-1)

rmse = mean_squared_error(y, oof) ** 0.5
mae_log = mean_absolute_error(y, oof)
mae_usd = mean_absolute_error(F["price"], np.exp(oof))

print(f"GroupKFold(5) by host  RMSE(log)={rmse:.3f}  MAE(log)={mae_log:.3f}  MAE($)=${mae_usd:,.0f}")

F = F.assign(
    fair_log_price=oof,
    fair_price=np.exp(oof),
    residual=y - oof,
)
F[["id", "price", "fair_price", "log_price", "fair_log_price", "residual"]].head()

## Save fair prices

OOF RF predictions as the listing-level fair prices. `fair_en` left empty until an
Elastic net city model exists; same csv schema so EN can fill that column later.

In [ ]:
out_dir = Path("..") / "data" / "fair_prices"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f"{CITY}_{SNAPSHOT}_fair_prices.csv"

fair = pd.DataFrame({
    "id": F["id"].values if "id" in F.columns else data.sample["id"].values,
    "city": CITY,
    "actual": F["price"].values,
    "fair_en": np.nan,  # fill later when EN city model exists
    "fair_rf": F["fair_price"].values,
    "residual_rf": F["residual"].values,  # log_price - fair_log_price; <0 → cheaper than RF fair
})
fair.to_csv(out_path, index=False)
print(f"wrote {len(fair):,} rows → {out_path.resolve()}")
fair.head()

## Fit on all data + feature importances

Refit once on the full sample for a quick importance view. Impurity-based (not causal).
For fold-safe rankings, prefer permutation importance in `05_feature_selection`.

In [ ]:
import matplotlib.pyplot as plt

model.fit(X, y)

# OrdinalEncoder keeps one column per categorical → names are num + cat
feat_names = list(num) + list(cat)
importances = pd.Series(
    model.named_steps["randomforestregressor"].feature_importances_,
    index=feat_names,
).sort_values(ascending=False)

top = importances.head(20)
ax = top.iloc[::-1].plot(kind="barh", figsize=(8, 6), title=f"Top 20 RF importances — {CITY} {SNAPSHOT}")
ax.set_xlabel("mean decrease in impurity")
plt.tight_layout()
plt.show()

importances.head(20).to_frame("importance")